# 05 - Signal Processing and Frequency Analysis

`epyr.signalprocessing` extracts oscillation frequencies from time-domain
pulse-EPR data (Rabi, DEER, HYSCORE) via FFT, with apodization and zero padding.

- `analyze_frequencies(t, y, ...)` -> a single dict of results
- `analyze_frequencies_2d(t, y, ...)` -> a 4-tuple `(freqs, axis, spectrum, info)`
- `power_spectrum(...)`, `spectrogram_analysis(...)` -> dicts

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")  # keep tutorial output readable

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import epyr

DATA = Path("..") / "data"   # example datasets, relative to this notebook
print("EPyR Tools version:", epyr.__version__)

In [ ]:
from epyr.signalprocessing import (
    analyze_frequencies, analyze_frequencies_2d, power_spectrum, spectrogram_analysis, apowin,
)

## 1D Rabi oscillation

A single Rabi trace: echo intensity versus pulse length.

In [ ]:
x, y, params, _ = epyr.eprload(DATA / "2024_08_CaWO4171Yb_rabi_6K_6724G_18dB.DTA",
                               plot_if_possible=False)
t_ns = x
sig = np.real(y)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t_ns, sig, color="C0")
ax.set_xlabel("Pulse length (ns)"); ax.set_title("Rabi oscillation")
plt.show()

res = analyze_frequencies(t_ns, sig, window="hann", zero_padding=4, plot=True)
plt.show()
print("dominant frequency:", res["dominant_frequencies"][0], res["freq_unit"])

## Apodization windows

Windowing trades frequency resolution against spectral leakage. `apowin`
generates the window functions directly.

In [ ]:
n = 256
fig, ax = plt.subplots(figsize=(8, 3))
for wtype in ("hann", "hamming", "blackman", "bartlett"):
    ax.plot(apowin(wtype, n), label=wtype)
ax.legend(); ax.set_title("Apodization windows"); ax.set_xlabel("sample")
plt.show()

## 2D Rabi: row-by-row FFT

A 2D Rabi dataset has one oscillation per row. `analyze_frequencies_2d` in
`row_by_row` mode returns the per-row spectra.

In [ ]:
x2, y2, p2, _ = epyr.eprload(DATA / "Rabi2D_GdCaWO4_13dB_3057G.DSC", plot_if_possible=False)
t2_ns = x2[0]
sig2 = np.real(y2)
print("2D Rabi shape:", sig2.shape)

freqs, axis, spectrum, info = analyze_frequencies_2d(
    t2_ns, sig2, mode="row_by_row", window="hann", zero_padding=2,
    plot_result=True, freq_range=(-50, 50))
plt.show()
print("frequency unit:", info["freq_unit"])

## Power spectral density and spectrogram

Applied to the averaged 2D Rabi trace.

In [ ]:
avg = sig2.mean(axis=0)

psd = power_spectrum(t2_ns, avg, method="welch", window="hann", nperseg=256, plot=True)
plt.show()

sg = spectrogram_analysis(t2_ns, avg, window="hann", nperseg=128, overlap=0.9, plot=True)
plt.show()

## Summary

- `analyze_frequencies` returns one dict; `analyze_frequencies_2d` returns a 4-tuple.
- Always remove the DC offset (done by default) and apply a window to reduce leakage.
- `power_spectrum` and `spectrogram_analysis` give PSD and time-frequency views.